In [13]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)



In [14]:
# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
misconception_mapping = pd.read_csv('misconception_mapping.csv')

# Display first few rows
train.head()


,QuestionId,ConstructId,ConstructName,SubjectId,SubjectName,CorrectAnswer,QuestionText,AnswerAText,AnswerBText,AnswerCText,AnswerDText,MisconceptionAId,MisconceptionBId,MisconceptionCId,MisconceptionDId
0,0,856,Use the order of operations to carry out calcu...,33,BIDMAS,A,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \),\( 3 \times(2+4-5) \),Does not need brackets,NaN,NaN,NaN,1672.0
1,1,1612,Simplify an algebraic fraction by factorising ...,1077,Simplifying Algebraic Fractions,D,"Simplify the following, if possible: \( \frac{...",\( m+1 \),\( m+2 \),\( m-1 \),Does not simplify,2142.0,143.0,2142.0,NaN
2,2,2774,Calculate the range from a list of data,339,Range and Interquartile Range from a List of Data,B,Tom and Katie are discussing the \( 5 \) plant...,Only\nTom,Only\nKatie,Both Tom and Katie,Neither is correct,1287.0,NaN,1287.0,1073.0
3,3,2377,Recall and use the intersecting diagonals prop...,88,Properties of Quadrilaterals,C,The angles highlighted on this rectangle with ...,acute,obtuse,\( 90^{\circ} \),Not enough information,1180.0,1180.0,NaN,1180.0
4,4,3387,Substitute positive integer values into formul...,67,Substitution into Formula,A,The equation \( f=3 r^{2}+3 \) is used to find...,\( 30 \),\( 27 \),\( 51 \),\( 24 \),NaN,NaN,NaN,1818.0


In [15]:
# Check for missing values
print("Train dataset missing values:\n", train.isnull().sum())

# Summary statistics
print(train.describe())

# Data types
print(train.info())


Train dataset missing values:
 QuestionId            0
ConstructId           0
ConstructName         0
SubjectId             0
SubjectName           0
CorrectAnswer         0
QuestionText          0
AnswerAText           0
AnswerBText           0
AnswerCText           0
AnswerDText           0
MisconceptionAId    734
MisconceptionBId    751
MisconceptionCId    789
MisconceptionDId    832
dtype: int64
        QuestionId  ConstructId    SubjectId  MisconceptionAId  \
count  1869.000000  1869.000000  1869.000000       1135.000000   
mean    934.000000  1613.261637   225.370787       1308.599119   
std     539.678145  1060.591804   238.536233        744.518370   
min       0.000000     4.000000    33.000000          1.000000   
25%     467.000000   575.000000    92.000000        686.000000   
50%     934.000000  1470.000000   203.000000       1336.000000   
75%    1401.000000  2637.000000   238.000000       1954.000000   
max    1868.000000  3526.000000  1984.000000       2585.000000   

 

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Extract question and answer texts
train_questions = train['QuestionText']
answer_texts = train[['AnswerAText', 'AnswerBText', 'AnswerCText', 'AnswerDText']]

# TF-IDF vectorization (example for QuestionText)
vectorizer = TfidfVectorizer(max_features=5000)
X_questions = vectorizer.fit_transform(train_questions)

print("TF-IDF features shape:", X_questions.shape)


TF-IDF features shape: (1869, 2315)


In [19]:
# Handle missing values in the target variable
valid_rows = ~train['MisconceptionAId'].isnull()  # Identify valid rows
X = X_questions[valid_rows].toarray()  # Filter corresponding rows in features
y = train.loc[valid_rows, 'MisconceptionAId']  # Filter valid rows in target

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a simple model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate
accuracy = model.score(X_val, y_val)
print(f"Validation Accuracy: {accuracy:.2f}")


Validation Accuracy: 0.13


In [20]:
# Predict on the test set
test_questions = test['QuestionText']
X_test_questions = vectorizer.transform(test_questions).toarray()
predictions = model.predict(X_test_questions)

# Create submission file
submission = pd.DataFrame({
    'QuestionId_Answer': test['QuestionId'].astype(str) + "_" + test['CorrectAnswer'],
    'MisconceptionId': predictions
})
submission.to_csv('submission.csv', index=False)

print("Submission file created!")


Submission file created!
